###Read xlsx bdx files from volumes

In [0]:
%sql
USE CATALOG bdx;
use schema data_dictionary;

In [0]:
%pip install openai openpyxl tiktoken
%pip install -U mlflow
dbutils.library.restartPython()

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/80/9a/f34f163294345f123673ed03e77c33dee2534f3ac1f9d18120384457304d/openai-1.75.0-py3-none-any.whl.metadata
  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for tiktoken from https://files.pythonhosted.org/packages/b1/73/41591c525680cd460a6becf56c9b17468d3711b1df242c53d2c7b2183d16/tiktoken-0.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for anyio<5,>=3.5.0 from https://files.pythonhosted.org/packages/a1/ee/48ca1a7c89ffec8b6a0c5d02b89c305671d5ffd8d3c94acf8b8c408575bb/anyio-4.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943e

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# ⚙️ Global Toggles for Pipeline Behavior
# These flags allow fine-grained control over the ingestion pipeline

# 🧠 GPT Logic Controls
ENABLE_GPT_CACHING = False         # Enable/disable schema caching for repeated headers
ENABLE_GPT_ENRICHMENT = True      # Use GPT to enrich column descriptions (binders glossary)
ENABLE_LANGCHAIN_GPT = False       # Use LangChain-based schema parsing (fallback to OpenAI if False)

# 🚀 Pipeline Toggles
USE_SEMANTIC_FILTERING = True            # Filter by GPT semantic score
ENABLE_HEADER_SHIFTING = True            # Auto-detect left padding/misaligned headers
ENABLE_COLUMN_PADDING = True             # Pad shorter rows to match header length
ENABLE_EXTRA_FIELD_CAPTURE = False        # Capture spillover/extra fields
semantic_score_threshold = 0.2           # Confidence cutoff
ENABLE_DELTA_WRITE = False   


# 📊 Thresholds
SEMANTIC_SCORE_THRESHOLD = 0.2    # Minimum score for semantic column filtering
gpt_schema_cache = {}

In [0]:
def get_langchain_gpt_schema_chain(client):
    """
    Constructs a LangChain chain using a detailed financial-specific schema prompt.
    """

    from langchain.chains.openai_functions import create_structured_output_chain
    from langchain.prompts import ChatPromptTemplate
    from langchain_core.utils.function_calling import convert_to_openai_function
    from langchain_core.output_parsers import JsonOutputKeyTools

    prompt = ChatPromptTemplate.from_template("""
You are an expert financial data analyst skilled at cleaning spreadsheet data.

{context}
You are analyzing a messy Excel or CSV sheet that was flattened into tab-separated text.

This sheet contains:
- Notes or metadata above the table
- Multiple stacked header rows (2 to 4 rows)
- Merged cells that split headers over several rows

Your job is to:
1. Identify the first row that contains real tabular data (ignore top notes)
2. Merge all relevant header rows into a single list of clear, standardized column headers
3. Clean and normalize header text to be consistent and meaningful

📌 Additionally:
- Use **standardized financial terminology** for column headers when possible.
- For example:
    - "CY2023 Est" → "2023_Estimate"
    - "Q1 2024" → "2024_Q1"
    - "2024 Av Commission" → "2024_Average_Commission"
    - "2024 Treaty Year Estimate" → "2024_Treaty_Year_Estimate"
    - "Class" → "Business_Line" or "Segment" if applicable

Return:
- "columns": a list of column metadata objects with:
    - original_column_name
    - standardized_column_name
    - description
    - inferred_data_type
    - percent_null
    - examples
    - semantic_score
- "column_headers": the cleaned, standardized header names in order
- "data_start_row": the 0-based row index where actual tabular data starts

Be smart and careful:
- Distinguish between notes vs headers vs data rows
- Resolve multi-row headers with merged cells intelligently
- Skip total/footer rows that are not part of the core data table

Text:
{prompt_text}
""")

    output_format = convert_to_openai_function(data_dict_function)
    chain = create_structured_output_chain(output_format, prompt, llm=client)
    return chain


In [0]:
from openai import OpenAI
from openai import AzureOpenAI
import os
import os
import json
import pandas as pd
from openai import OpenAI

endpoint = "https://hanna-m9ic9273-eastus2.cognitiveservices.azure.com/"
model_name = "gpt-4.1"
deployment = "gpt-4.1"

subscription_key = "C9oe9lxteRbZXvRHdkXRq7uezqsl1bDWQ7tJH91uAHaWbjnpoQ8rJQQJ99BDACHYHv6XJ3w3AAAAACOGb14W"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [0]:
def clean_percent(val):
    """
    Converts a value like '45.8%' or ' 22 % ' into a float.
    Returns None for invalid or missing entries.
    """
    try:
        if isinstance(val, str):
            val = val.replace("%", "").strip()
        return float(val)
    except (ValueError, TypeError):
        return None

In [0]:
from collections import defaultdict
import pandas as pd
from pathlib import Path

def gpt_flatten(df, sheet_text, client, file_name, sheet_name):
    """
    GPT-guided header reconstruction + data extraction.
    """
    result = generate_rich_data_dictionary(sheet_text, client, file_name, sheet_name)
    headers = result.get("column_headers", [])
    data_start = result.get("data_start_row", 0)

    data_rows = df.iloc[data_start:].reset_index(drop=True)

    if data_rows.empty or len(headers) != data_rows.shape[1]:
        raise ValueError("Header mismatch or no usable rows")

    data_rows.columns = headers
    flat_text = "\n".join([
        "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
        for row in data_rows.values
    ])
    return flat_text

def fallback_flatten(df):
    """
    Simple fallback: raw tab-separated flattening.
    """
    return "\n".join([
        "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
        for row in df.values if any(pd.notna(row))
    ])

def flatten_xlsx_from_volume(volume_folder, client):
    """
    Reads Excel files from a Unity Catalog volume and flattens them.
    Includes GPT-based flattening and fallback mode.
    """
    all_text_blocks = {}
    flatten_stats = defaultdict(list)

    try:
        full_paths = [
            f.path for f in dbutils.fs.ls(volume_folder)
            if f.path.lower().endswith(".xlsx")
        ]
    except Exception as e:
        print(f"❌ Failed to list files in {volume_folder} — {e}")
        return all_text_blocks

    if not full_paths:
        print(f"ℹ️ No Excel files found in {volume_folder}")
        return all_text_blocks

    print(f"📂 Found {len(full_paths)} Excel files")

    for dbfs_path in full_paths:
        file_name = Path(dbfs_path).name
        print(f"\n📥 Processing: {file_name}")

        try:
            local_path = copy_volume_file_to_tmp_via_spark(dbfs_path)
            with pd.ExcelFile(local_path) as xls:
                for sheet in xls.sheet_names:
                    key = f"{file_name}::{sheet}"
                    try:
                        df = pd.read_excel(xls, sheet_name=sheet, header=None)

                        if df.empty or df.shape[1] < MIN_COLUMNS_REQUIRED:
                            print(f"⚠️ Skipping malformed sheet: {sheet}")
                            flatten_stats["skipped"].append(key)
                            continue

                        # Convert to raw text
                        sheet_text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in df.values if any(pd.notna(row))
                        ])

                        if USE_GPT_FOR_FLATTENING:
                            try:
                                flattened = gpt_flatten(df, sheet_text, client, file_name, sheet)
                                all_text_blocks[key] = flattened
                                flatten_stats["structured"].append(key)
                                print(f"✅ GPT flatten: {key} ({df.shape[0]} rows)")
                            except Exception as gpt_error:
                                print(f"  ⚠️ GPT failed: {gpt_error}")
                                if ALLOW_FALLBACK_FLATTENING:
                                    flattened = fallback_flatten(df)
                                    all_text_blocks[key] = flattened
                                    flatten_stats["fallback"].append(key)
                                    print(f"  ✅ Fallback flatten: {key}")
                                else:
                                    flatten_stats["failed"].append(key)
                        else:
                            # Only fallback
                            flattened = fallback_flatten(df)
                            all_text_blocks[key] = flattened
                            flatten_stats["fallback"].append(key)
                            print(f"✅ Fallback flatten: {key}")

                    except Exception as sheet_error:
                        print(f"❌ Sheet-level error for {key}: {sheet_error}")
                        flatten_stats["failed"].append(key)

        except Exception as file_error:
            print(f"❌ File-level error for {file_name}: {file_error}")
            flatten_stats["failed"].append(file_name)

    print("\n📊 Flattening Summary")
    print(f"  ✅ Structured: {len(flatten_stats['structured'])}")
    print(f"  🔁 Fallback  : {len(flatten_stats['fallback'])}")
    print(f"  ⚠️ Skipped   : {len(flatten_stats['skipped'])}")
    print(f"  ❌ Failed    : {len(flatten_stats['failed'])}")

    return all_text_blocks


In [0]:
# def extract_headers_and_data_start(raw_text, client):
#     """
#     Uses GPT to infer structured headers and the starting row for actual data
#     from messy Excel text.
    
#     Args:
#         raw_text (str): Flattened text from Excel sheet
#         client: OpenAI client
    
#     Returns:
#         dict: {
#             'column_headers': [...],
#             'data_start_row': int
#         }
#     """
#     prompt = f"""
# You are given messy tab-separated text from an Excel sheet. It may contain multiple rows of headers,
# notes, or descriptive text before the actual data begins.

# Your job is to:
# 1. Identify the clean, final list of column headers.
# 2. Indicate which row (0-indexed) the actual data starts on.

# Text (first 8000 chars only):
# {raw_text[:8000]}

# Return JSON with keys:
# - column_headers: list of strings
# - data_start_row: integer (first row of actual data)
# """

#     response = client.chat.completions.create(
#         model="gpt-4.1",
#         messages=[{"role": "user", "content": prompt}]
#     )

#     # Attempt to parse clean JSON response
#     try:
#         return json.loads(response.choices[0].message.content)
#     except Exception as e:
#         print(f"⚠️ Failed to parse GPT response: {e}")
#         return {
#             "column_headers": [],
#             "data_start_row": 0
#         }


In [0]:
def gpt_guided_dataframe_from_xlsx(file_path: str, sheet_name: str, raw_text: str, client, debug=False) -> pd.DataFrame:
    import pandas as pd
    from collections import Counter
    import numpy as np
    import re

    df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    raw_rows = df_raw.values.tolist()

    # === Step 1: GPT schema inference ===
    schema = generate_rich_data_dictionary(raw_text, client)
    gpt_headers = schema.get("column_headers", [])
    data_start_row = schema.get("data_start_row", 0)
    col_count = len(gpt_headers)

    if debug:
        print(f"🧠 GPT headers: {gpt_headers}")
        print(f"🟢 Data starts at row: {data_start_row}")

    # === Step 2: Slice rows GPT believes are data ===
    data_rows = raw_rows[data_start_row:]
    if not data_rows:
        raise ValueError("❌ No data rows found after GPT start row.")

    # === Step 3: Smart alignment using blank offset ===
    def leading_blanks(row):
        return next((i for i, cell in enumerate(row) if str(cell).strip()), len(row))

    offset_counts = Counter([leading_blanks(r) for r in data_rows if any(pd.notna(r))])
    common_offset = offset_counts.most_common(1)[0][0] if offset_counts else 0

    if debug and ENABLE_HEADER_SHIFTING:
        print(f"🔍 Most common shift: {common_offset}")

    aligned = []
    for row in data_rows:
        if ENABLE_HEADER_SHIFTING:
            trimmed = row[common_offset:]
        else:
            trimmed = row

        if ENABLE_COLUMN_PADDING:
            trimmed = trimmed[:col_count] + [""] * max(0, col_count - len(trimmed))

        aligned.append([str(cell).strip() if cell is not None else "" for cell in trimmed])

    # === Step 4: Final DataFrame ===
    df_aligned = pd.DataFrame(aligned, columns=gpt_headers)

    if df_aligned.shape[1] != col_count:
        raise ValueError("❌ Column count mismatch after alignment.")

    return df_aligned


In [0]:
# 📦 Caching GPT responses to avoid duplicate costs
from functools import wraps
from hashlib import sha256

def gpt_cache(func):
    @wraps(func)
    def wrapper(raw_text, *args, **kwargs):
        lines = raw_text.strip().split("\n")
        header_sample = "\n".join(lines[:6] + lines[6:16])
        key = sha256(header_sample.encode()).hexdigest()

        if ENABLE_GPT_CACHING and key in gpt_schema_cache:
            print("♻️ [Cache Hit] Reusing GPT schema result")
            return gpt_schema_cache[key]

        result = func(raw_text, *args, **kwargs)

        if ENABLE_GPT_CACHING:
            gpt_schema_cache[key] = result
            print("🧠 [Cache Miss] GPT schema generated and cached")
        return result
    return wrapper


In [0]:
data_dict_function = {
    "name": "generate_rich_data_dictionary",
    "description": "Analyze raw Excel/CSV sheet and return standardized column metadata and structural layout.",
    "parameters": {
        "type": "object",
        "properties": {
            "columns": {
                "type": "array",
                "description": "List of detected columns with metadata",
                "items": {
                    "type": "object",
                    "properties": {
                        "original_column_name": {"type": "string"},
                        "standardized_column_name": {"type": "string"},
                        "description": {"type": "string"},
                        "inferred_data_type": {"type": "string"},
                        "percent_null": {"type": "number"},
                        "examples": {"type": "array", "items": {"type": "string"}},
                        "semantic_score": {"type": "number"}
                    },
                    "required": [
                        "original_column_name",
                        "standardized_column_name",
                        "description",
                        "inferred_data_type",
                        "percent_null",
                        "examples",
                        "semantic_score"
                    ]
                }
            },
            "column_headers": {
                "type": "array",
                "description": "Final cleaned and standardized column headers in order",
                "items": {"type": "string"}
            },
            "data_start_row": {
                "type": "integer",
                "description": "Row number (0-indexed) where actual data starts"
            }
        },
        "required": ["columns", "column_headers", "data_start_row"]
    }
}



In [0]:
# @gpt_cache
# def generate_rich_data_dictionary(raw_text, client, file_name=None, sheet_name=None):
#     from hashlib import sha256
#     import json

#     # === Step 1: Token-efficient sampling ===
#     lines = raw_text.strip().split("\n")
#     header_lines = lines[:6]
#     sample_data_lines = lines[6:16]
#     prompt_text = "\n".join(header_lines + sample_data_lines)

#     # === Step 2: Caching ===
#     header_hash = sha256(prompt_text.encode()).hexdigest()
#     if header_hash in gpt_schema_cache:
#         return gpt_schema_cache[header_hash]

#     # === Step 3: Add metadata context ===
#     context = ""
#     if file_name or sheet_name:
#         context += f"This file is named '{file_name}' and sheet is '{sheet_name}'. It may contain financial, tabular, or projection-related data.\n"

#     # === Step 4: Optimized GPT Prompt ===
#     prompt = f"""
# You are an expert financial data analyst skilled at cleaning spreadsheet data.

# {context}
# You are analyzing a messy Excel or CSV sheet that was flattened into tab-separated text.

# This sheet contains:
# - Notes or metadata above the table
# - Multiple stacked header rows (2 to 4 rows)
# - Merged cells that split headers over several rows

# Your job is to:
# 1. Identify the first row that contains real tabular data (ignore top notes)
# 2. Merge all relevant header rows into a single list of clear, standardized column headers
# 3. Clean and normalize header text to be consistent and meaningful

# 📌 Additionally:
# - Use **standardized financial terminology** for column headers when possible.
# - For example:
#     - "CY2023 Est" → "2023_Estimate"
#     - "Q1 2024" → "2024_Q1"
#     - "2024 Av Commission" → "2024_Average_Commission"
#     - "2024 Treaty Year Estimate" → "2024_Treaty_Year_Estimate"
#     - "Class" → "Business_Line" or "Segment" if applicable

# Return:
# - "columns": a list of column metadata objects with:
#     - original_column_name
#     - standardized_column_name
#     - description
#     - inferred_data_type
#     - percent_null
#     - examples
#     - semantic_score
# - "column_headers": the cleaned, standardized header names in order
# - "data_start_row": the 0-based row index where actual tabular data starts

# Be smart and careful:
# - Distinguish between notes vs headers vs data rows
# - Resolve multi-row headers with merged cells intelligently
# - Skip total/footer rows that are not part of the core data table
# Text:
# {prompt_text}
# """

#     try:
#         response = client.chat.completions.create(
#             model="gpt-4.1",
#             messages=[{"role": "user", "content": prompt}],
#             functions=[data_dict_function],
#             function_call={"name": "generate_rich_data_dictionary"},
#             temperature=0.0  # ⬅️ deterministic output
#         )

#         result = json.loads(response.choices[0].message.function_call.arguments)
#         gpt_schema_cache[header_hash] = result  # 🧠 Cache result
#         return result

#     except Exception as e:
#         print(f"❌ GPT schema extraction failed: {e}")
#         return {
#             "columns": [],
#             "column_headers": [],
#             "data_start_row": 0
#         }


In [0]:
@gpt_cache
def generate_rich_data_dictionary(raw_text, client, file_name=None, sheet_name=None):
    import json

    # === Step 1: Token-efficient prompt sampling ===
    lines = raw_text.strip().split("\n")
    header_lines = lines[:6]
    sample_data_lines = lines[6:16]
    prompt_text = "\n".join(header_lines + sample_data_lines)

    # === Step 2: Optional metadata context ===
    context = ""
    if file_name or sheet_name:
        context += f"This file is named '{file_name}' and sheet is '{sheet_name}'. It may contain financial, tabular, or projection-related data.\n"

    # === Step 3: LangChain optional logic ===
    if ENABLE_LANGCHAIN_GPT:
        try:
            chain = get_langchain_gpt_schema_chain(client)
            result = chain.invoke({"raw_text": prompt_text})
            print("🔗 LangChain used for GPT schema.")
            return result
        except Exception as e:
            print(f"⚠️ LangChain failed — falling back to raw OpenAI call: {e}")

    # === Step 4: Optimized GPT Prompt ===
    prompt = f"""
You are an expert financial data analyst skilled at cleaning spreadsheet data.

{context}
You are analyzing a messy Excel or CSV sheet that was flattened into tab-separated text.

This sheet contains:
- Notes or metadata above the table
- Multiple stacked header rows (2 to 4 rows)
- Merged cells that split headers over several rows

Your job is to:
1. Identify the first row that contains real tabular data (ignore top notes)
2. Merge all relevant header rows into a single list of clear, standardized column headers
3. Clean and normalize header text to be consistent and meaningful

📌 Additionally:
- Use **standardized financial terminology** for column headers when possible.
- For example:
    - "CY2023 Est" → "2023_Estimate"
    - "Q1 2024" → "2024_Q1"
    - "2024 Av Commission" → "2024_Average_Commission"
    - "2024 Treaty Year Estimate" → "2024_Treaty_Year_Estimate"
    - "Class" → "Business_Line" or "Segment" if applicable

Return:
- "columns": a list of column metadata objects with:
    - original_column_name
    - standardized_column_name
    - description
    - inferred_data_type
    - percent_null
    - examples
    - semantic_score
- "column_headers": the cleaned, standardized header names in order
- "data_start_row": the 0-based row index where actual tabular data starts

Be smart and careful:
- Distinguish between notes vs headers vs data rows
- Resolve multi-row headers with merged cells intelligently
- Skip total/footer rows that are not part of the core data table
Text:
{prompt_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            functions=[data_dict_function],
            function_call={"name": "generate_rich_data_dictionary"},
            temperature=0.0
        )

        return json.loads(response.choices[0].message.function_call.arguments)

    except Exception as e:
        print(f"❌ GPT schema extraction failed: {e}")
        return {
            "columns": [],
            "column_headers": [],
            "data_start_row": 0
        }


In [0]:
import os
from pathlib import Path

def copy_volume_file_to_tmp_via_spark(volume_path: str) -> str:
    """
    Copies a file from Unity Catalog volume to /tmp/ using Spark's binaryFile format.

    Args:
        volume_path (str): Full DBFS/Volume path to the file

    Returns:
        str: Local /tmp path where the file is saved

    Raises:
        FileNotFoundError: If the file doesn't exist at the given volume path
        IOError: If there's an error reading or writing the file
    """
    file_name = Path(volume_path).name
    tmp_path = f"/tmp/{file_name}"

    # Validate file existence
    try:
        dbutils.fs.ls(volume_path)
    except Exception:
        raise FileNotFoundError(f"🚫 File not found: {volume_path}")

    try:
        binary_df = spark.read.format("binaryFile").load(volume_path)
        content_rows = binary_df.select("content").collect()

        if not content_rows or not content_rows[0]["content"]:
            raise IOError(f"⚠️ File is empty or unreadable: {volume_path}")

        with open(tmp_path, "wb") as f:
            f.write(content_rows[0]["content"])

        print(f"✅ File copied to: {tmp_path}")
        return tmp_path

    except Exception as e:
        raise IOError(f"❌ Failed to copy from volume: {volume_path} — {str(e)}")


In [0]:
def flatten_xlsx_from_volume(volume_folder: str, use_gpt_schema: bool = True, client=None):
    """
    Flattens all Excel sheets from the given volume.
    Supports GPT-guided header detection or raw fallback.
    
    Returns:
        dict: {filename::sheet_name: tab-separated cleaned text}
    """
    import pandas as pd
    from pathlib import Path
    from collections import defaultdict

    all_text_blocks = {}
    flatten_stats = defaultdict(list)

    try:
        full_paths = [
            f.path for f in dbutils.fs.ls(volume_folder)
            if f.name.lower().endswith(".xlsx")
        ]
    except Exception as e:
        print(f"❌ Failed to list files in {volume_folder} — {e}")
        return all_text_blocks

    print(f"📊 Found {len(full_paths)} Excel files")

    for dbfs_path in full_paths:
        file_name = Path(dbfs_path).name
        print(f"📂 {file_name}")

        try:
            local_path = copy_volume_file_to_tmp_via_spark(dbfs_path)
            xls = pd.ExcelFile(local_path)

            for sheet in xls.sheet_names:
                sheet_key = f"{file_name}::{sheet}"
                try:
                    df = pd.read_excel(xls, sheet_name=sheet, header=None)

                    # === Use GPT for intelligent flattening ===
                    if use_gpt_schema and client:
                        sheet_text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in df.values if any(pd.notna(row))
                        ])

                        result = generate_rich_data_dictionary(sheet_text, client, file_name, sheet)
                        headers = result.get("column_headers", [])
                        start_row = result.get("data_start_row", 0)

                        sliced = df.iloc[start_row:].reset_index(drop=True)
                        if sliced.empty or len(headers) != sliced.shape[1]:
                            raise ValueError("Mismatch between headers and data shape")

                        sliced.columns = headers
                        text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in sliced.values
                        ])
                        flatten_stats["gpt"].append(sheet_key)

                    else:
                        # === Raw fallback ===
                        text = "\n".join([
                            "\t".join("" if pd.isna(cell) else str(cell).strip() for cell in row)
                            for row in df.values if any(pd.notna(row))
                        ])
                        flatten_stats["raw"].append(sheet_key)

                    all_text_blocks[sheet_key] = text
                    print(f"  ✅ Flattened: {sheet_key}")

                except Exception as e_sheet:
                    print(f"  ❌ Failed: {sheet_key} — {e_sheet}")
                    flatten_stats["failed"].append(sheet_key)

        except Exception as e_file:
            print(f"❌ File error {file_name} — {e_file}")

    # Summary
    print("\n📊 Flattening Summary:")
    for category in flatten_stats:
        print(f"  {category.upper():<8}: {len(flatten_stats[category])}")

    return all_text_blocks


In [0]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql import DataFrame

# Fully qualify if working in Unity Catalog
# schema_table_name = "bdx.data_dictionary.gpt_data_dictionary"  # or 'catalog.schema.gpt_data_dictionary'

# # Flag for tracking table state
# table_created = False


In [0]:
# from datetime import datetime

# def process_data_dictionaries(raw_text_map, schema_table_name):
#     """
#     Processes raw text data into structured data dictionaries and writes to a Delta table.
    
#     Args:
#         raw_text_map (dict): Dictionary of {sheet_key: raw_text}
#         schema_table_name (str): Full name of the Delta table (e.g., 'bdx.cleaned_schema')

#     Returns:
#         dict: Map of processed data dictionaries, keyed by sheet_key
#     """
#     processed_data_dictionary_map = {}
#     total_sheets = len(raw_text_map)
#     processed_count = 0
    
#     print(f"🔍 Processing {total_sheets} sheets to extract data dictionaries")

#     # Check if Delta table already exists
#     table_exists = False
#     try:
#         spark.sql("DESCRIBE TABLE bdx.data_dictionary.gpt_data_dictionary")
#         table_exists = True
#         print(f"📊 Table {schema_table_name} exists — will update entries")
#     except:
#         print(f"📊 Table {schema_table_name} will be created")

#     # Buffer for batch insertion
#     all_flattened_records = []

#     for sheet_key, raw_text in raw_text_map.items():
#         print(f"\n🧠 Extracting schema for: {sheet_key}")
#         try:
#             # Step 1: Extract metadata using LLM
#             data_dictionary = generate_rich_data_dictionary(raw_text,client)
#             processed_data_dictionary_map[sheet_key] = data_dictionary

#             # Step 2: Flatten structure
#             sheet_records = [
#             {
#                 "file_name": sheet_key.split("::")[0],
#                 "sheet_name": sheet_key.split("::")[1],
#                 "sheet_key": sheet_key,
#                 "column_name": col.get("column_name"),
#                 "description": col.get("description"),
#                 "inferred_data_type": col.get("inferred_data_type"),
#                 "percent_null": clean_percent(col.get("percent_null")),  # ✅ fixed
#                 "examples": ", ".join(str(x) for x in col.get("examples", [])),
#                 "semantic_score": col.get("semantic_score"),
#                 "processed_at": str(datetime.utcnow().isoformat())
#             }
#             for col in data_dictionary.get("columns", [])
#         ]


#             all_flattened_records.extend(sheet_records)
#             processed_count += 1
#             print(f"✅ Processed {sheet_key}: {len(sheet_records)} columns")

#         except Exception as e:
#             print(f"❌ Failed on {sheet_key} — {str(e)}")

#     # Step 3: Save to Delta if records found
#     from pyspark.sql.types import DoubleType
#     if all_flattened_records:
#         try:
#             # Convert to Spark DataFrame
#             new_data_df = spark.createDataFrame(pd.DataFrame(all_flattened_records))

#             # 🔧 Fix: Enforce type consistency for 'percent_null'
#             if "percent_null" in new_data_df.columns:
#                 new_data_df = new_data_df.withColumn("percent_null", new_data_df["percent_null"].cast(DoubleType()))

#             if table_exists:
#                 existing_df = spark.table(schema_table_name)
#                 sheet_keys = [r["sheet_key"] for r in all_flattened_records]
#                 filtered_existing = existing_df.filter(~existing_df.sheet_key.isin(sheet_keys))
#                 final_df = filtered_existing.unionByName(new_data_df)
#                 final_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)
#             else:
#                 new_data_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)

#             print(f"\n✅ Saved {processed_count}/{total_sheets} sheets to: {schema_table_name}")

#         except Exception as e:
#             print(f"\n❌ Failed writing to Delta table: {e}")
#             raise
#     else:
#         print("\n⚠️ No valid records to write")

#     return processed_data_dictionary_map


In [0]:
from datetime import datetime
from pyspark.sql.types import DoubleType
import pandas as pd


 # ⬅️ Toggle to skip actual Delta table write
semantic_score_threshold = 0.2

def clean_percent(val):
    try:
        if isinstance(val, str):
            val = val.replace("%", "").strip()
        return float(val)
    except:
        return None

def process_data_dictionaries(raw_text_map, schema_table_name):
    """
    Extracts GPT data dictionaries and writes to Delta (if enabled).
    Returns structured dictionary map.
    """
    processed_data_dictionary_map = {}
    all_flattened_records = []
    total_sheets = len(raw_text_map)
    processed_count = 0

    print(f"🔍 Processing {total_sheets} sheets")

    # Check table existence
    table_exists = False
    try:
        spark.sql(f"DESCRIBE TABLE {schema_table_name}")
        table_exists = True
        print(f"📊 Table exists: {schema_table_name}")
    except:
        print(f"📊 Will create new table: {schema_table_name}")

    for sheet_key, raw_text in raw_text_map.items():
        print(f"\n🧠 Extracting schema: {sheet_key}")
        try:
            # === Step 1: GPT schema extraction
            data_dictionary = generate_rich_data_dictionary(raw_text, client)
            processed_data_dictionary_map[sheet_key] = data_dictionary

            # === Step 2: Flatten columns
            seen = set()
            unique_columns = [
                col for col in data_dictionary.get("columns", [])
                if (col_id := col.get("standardized_column_name", col.get("original_column_name", ""))) not in seen and not seen.add(col_id)
            ]

            sheet_records = [
                {
                    "file_name": sheet_key.split("::")[0],
                    "sheet_name": sheet_key.split("::")[1],
                    "sheet_key": sheet_key,
                    "original_column_name": col.get("original_column_name", col.get("column_name", "")),
                    "standardized_column_name": col.get("standardized_column_name", col.get("column_name", "")),
                    "description": col.get("description", ""),
                    "description_enriched": col.get("description_enriched", "") if ENABLE_GPT_ENRICHMENT else "",
                    "inferred_data_type": col.get("inferred_data_type", "unknown"),
                    "percent_null": clean_percent(col.get("percent_null", 0)),
                    "examples": ", ".join(str(x) for x in col.get("examples", [])),
                    "semantic_score": col.get("semantic_score", None),
                    "processed_at": str(datetime.utcnow().isoformat())
                }
                for col in unique_columns
            ]

            all_flattened_records.extend(sheet_records)
            processed_count += 1
            print(f"✅ Extracted {len(sheet_records)} columns")

        except Exception as e:
            print(f"❌ Failed: {sheet_key} — {str(e)}")

    # === Step 3: Save to Delta
    if ENABLE_DELTA_WRITE and all_flattened_records:
        try:
            df_new = pd.DataFrame(all_flattened_records)
            spark_df = spark.createDataFrame(df_new)

            if "percent_null" in spark_df.columns:
                spark_df = spark_df.withColumn("percent_null", spark_df["percent_null"].cast(DoubleType()))

            if table_exists:
                existing_df = spark.table(schema_table_name)
                sheet_keys = [r["sheet_key"] for r in all_flattened_records]
                final_df = (
                    existing_df.filter(~existing_df.sheet_key.isin(sheet_keys))
                    .unionByName(spark_df)
                )
                final_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)
            else:
                spark_df.write.format("delta").mode("overwrite").saveAsTable(schema_table_name)

            print(f"\n✅ Delta table updated: {schema_table_name}")

        except Exception as e:
            print(f"\n❌ Failed writing to Delta: {e}")
            print("🧪 Schema of failed DataFrame:")
            spark_df.printSchema()

    elif not ENABLE_DELTA_WRITE:
        print("\n🧪 Skipped Delta write (dry run mode)")
    else:
        print("\n⚠️ No records to write")

    return processed_data_dictionary_map


In [0]:
volume_folder = "dbfs:/Volumes/test/bronze/raw/"
raw_text_map = flatten_xlsx_from_volume(volume_folder)

📊 Found 1 Excel files
📂 Japanese_CHAR.xlsx
✅ File copied to: /tmp/Japanese_CHAR.xlsx
  ✅ Flattened: Japanese_CHAR.xlsx::1.Statistics
  ✅ Flattened: Japanese_CHAR.xlsx::Statistics Edited-USE
  ✅ Flattened: Japanese_CHAR.xlsx::2 Buyer ex 200M
  ✅ Flattened: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1
  ✅ Flattened: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 2
  ✅ Flattened: Japanese_CHAR.xlsx::3 Special Limit for 2024
  ✅ Flattened: Japanese_CHAR.xlsx::4 Risk Profile
  ✅ Flattened: Japanese_CHAR.xlsx::5 Special Acceptence

📊 Flattening Summary:
  RAW     : 8


In [0]:
schema_table_name = "gpt_data_dictionary"
processed_data_dictionary_map = process_data_dictionaries(raw_text_map, schema_table_name)

🔍 Processing 8 sheets
📊 Table exists: gpt_data_dictionary

🧠 Extracting schema: Japanese_CHAR.xlsx::1.Statistics
✅ Extracted 8 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::Statistics Edited-USE
✅ Extracted 8 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::2 Buyer ex 200M
✅ Extracted 12 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1
✅ Extracted 13 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 2
✅ Extracted 12 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::3 Special Limit for 2024
✅ Extracted 14 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::4 Risk Profile
✅ Extracted 12 columns

🧠 Extracting schema: Japanese_CHAR.xlsx::5 Special Acceptence
✅ Extracted 0 columns

🧪 Skipped Delta write (dry run mode)


[Trace(request_id=tr-d940f8949ae84f13a9a105343c46fb8f), Trace(request_id=tr-2f3a5e92e41544a7bfcc8a64dad020c9), Trace(request_id=tr-95c3490f2fe54e3b85edbb9fc90195d3), Trace(request_id=tr-40151103157441039af968211d142554), Trace(request_id=tr-1706bc3450d947eb8e7205c4d7775d7a), Trace(request_id=tr-9e336c34bf1b438bbfaad787bee9eabf), Trace(request_id=tr-b2ccd30ab7914f19a21750dfb41af0aa), Trace(request_id=tr-b79f3b115ab445b697958ace1c2b2894)]

In [0]:
# === Toggles ===
ENABLE_VERBOSE_LOGGING = True
ENABLE_RETRY = True

# === Safe Init ===
if "processed_data_dictionary_map" not in globals():
    processed_data_dictionary_map = {}

failed_sheets = []
permanently_failed_sheets = []

def run_schema_extraction(sheet_key, raw_text, client, force_mode=False):
    try:
        if force_mode:
            raw_text = f"WARNING: Previous GPT parse failed. Force structured schema.\n\n{raw_text}"

        result = generate_rich_data_dictionary(raw_text, client)
        processed_data_dictionary_map[sheet_key] = result

        if ENABLE_VERBOSE_LOGGING:
            print(f"✅ Extracted schema for: {sheet_key}")
        return True

    except Exception as e:
        if ENABLE_VERBOSE_LOGGING:
            print(f"❌ Extraction failed for {sheet_key} — {str(e)}")
        return False

# === Main Loop ===
for sheet_key, raw_text in raw_text_map.items():
    if sheet_key in processed_data_dictionary_map:
        continue

    if ENABLE_VERBOSE_LOGGING:
        print(f"\n🧠 Extracting schema for: {sheet_key}")

    success = run_schema_extraction(sheet_key, raw_text, client)

    if not success and ENABLE_RETRY:
        if ENABLE_VERBOSE_LOGGING:
            print(f"🔁 Retrying with enhanced prompt for: {sheet_key}")
        retry_success = run_schema_extraction(sheet_key, raw_text, client, force_mode=True)

        if not retry_success:
            permanently_failed_sheets.append(sheet_key)


In [0]:
from typing import List
import json

def enrich_column_descriptions(columns: List[dict], client) -> List[dict]:
    """
    If ENABLE_GPT_ENRICHMENT is True:
        Enrich column descriptions using GPT in batch mode, fallback to per-column if batch fails.
    Else:
        Return columns as-is.
    """
    if not ENABLE_GPT_ENRICHMENT:
        print("⚙️ GPT enrichment is disabled — skipping.")
        return columns

    if not columns:
        return columns

    enriched = []

    # === 1. Batch prompt ===
    try:
        descriptions = [col["description"] for col in columns]
        numbered = [f"{i+1}. {desc}" for i, desc in enumerate(descriptions)]
        prompt = f"""
You are a financial data expert.

Improve each of the following column descriptions using standardized financial, delegated authority, and insurance terminology. 
Respond as a JSON array of improved strings, in the same order:

{chr(10).join(numbered)}
        """.strip()

        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )

        enriched_descriptions = json.loads(response.choices[0].message.content)

        if len(enriched_descriptions) != len(columns):
            raise ValueError("Mismatch in enriched output length.")

        for col, enriched_text in zip(columns, enriched_descriptions):
            col["description_enriched"] = enriched_text
            enriched.append(col)

        print(f"✅ Batch enrichment completed for {len(columns)} columns")

    except Exception as batch_error:
        print(f"⚠️ Batch enrichment failed: {batch_error}")
        print("🔁 Falling back to per-column GPT enrichment")

        # === 2. Fallback: Per-column enrichment ===
        for col in columns:
            try:
                prompt = f"Improve this data column description using financial glossary:\n\n'{col['description']}'"
                response = client.chat.completions.create(
                    model="gpt-4.1",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                new_desc = response.choices[0].message.content.strip()
                col["description_enriched"] = new_desc
            except Exception as e:
                col["description_enriched"] = col["description"]
                print(f"❌ GPT failed for column '{col.get('column_name', '')}': {e}")
            enriched.append(col)

    return enriched


In [0]:
# ✅ Apply GPT enrichment only if enabled
if ENABLE_GPT_ENRICHMENT:
    for sheet_key, schema in processed_data_dictionary_map.items():
        columns = schema.get("columns", [])
        enriched = enrich_column_descriptions(columns, client)
        processed_data_dictionary_map[sheet_key]["columns"] = enriched

    print(f"✅ Column descriptions enriched for {len(processed_data_dictionary_map)} sheets.")
else:
    print("⚙️ Skipping GPT enrichment (ENABLE_GPT_ENRICHMENT=False)")

⚠️ Batch enrichment failed: Expecting value: line 1 column 1 (char 0)
🔁 Falling back to per-column GPT enrichment
✅ Batch enrichment completed for 8 columns
⚠️ Batch enrichment failed: Expecting value: line 1 column 1 (char 0)
🔁 Falling back to per-column GPT enrichment
⚠️ Batch enrichment failed: Expecting value: line 1 column 1 (char 0)
🔁 Falling back to per-column GPT enrichment
⚠️ Batch enrichment failed: Expecting value: line 1 column 1 (char 0)
🔁 Falling back to per-column GPT enrichment
⚠️ Batch enrichment failed: Expecting value: line 1 column 1 (char 0)
🔁 Falling back to per-column GPT enrichment
✅ Batch enrichment completed for 12 columns
✅ Column descriptions enriched for 8 sheets.


[Trace(request_id=tr-fe42c508a0dc4828bd2f5de545b817b4), Trace(request_id=tr-ab9b274feefd45128c19c5ce09839458), Trace(request_id=tr-5621c4017d2e46298fa912bc688ef983), Trace(request_id=tr-10f8629a4b854ea89dbef605c4799d33), Trace(request_id=tr-c805ab7555414486b53d2149b4e0e563), Trace(request_id=tr-efbff6ab16fe4b8889be478252b60364), Trace(request_id=tr-bc9bb9be060d47e9b889a467f2a896c7), Trace(request_id=tr-036ab1bed85a45b1b97f013646e0ef64), Trace(request_id=tr-4f10b573d49b429f8815b09a9d11b432), Trace(request_id=tr-6c41159339084591a918aa8d9c251363)]

In [0]:
def gpt_smart_dataframe_from_xlsx(file_path: str, sheet_name: str, raw_text: str, client, debug=False) -> pd.DataFrame:
    import pandas as pd
    from collections import Counter
    import numpy as np
    import re
    from hashlib import sha256

    # === Step 1: GPT schema inference (with caching) ===
    lines = raw_text.strip().split("\n")
    sample_text = "\n".join(lines[:16])

    if ENABLE_GPT_CACHING:
        header_hash = sha256(sample_text.encode()).hexdigest()
        if header_hash in gpt_schema_cache:
            schema = gpt_schema_cache[header_hash]
            if debug: print("♻️ Using cached GPT schema")
        else:
            schema = generate_rich_data_dictionary(raw_text, client)
            gpt_schema_cache[header_hash] = schema
            if debug: print("🧠 GPT schema generated and cached")
    else:
        schema = generate_rich_data_dictionary(raw_text, client)
        if debug: print("🧠 GPT schema generated without cache")

    gpt_headers = schema.get("column_headers", [])
    data_start_row = schema.get("data_start_row", 0)
    col_count = len(gpt_headers)

    if debug:
        print(f"📊 Columns: {col_count} | Data starts at: {data_start_row}")

    # === Step 2: Load Excel file ===
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    except Exception as e:
        raise RuntimeError(f"❌ Failed to read Excel: {e}")

    raw_rows = df_raw.values.tolist()
    data_rows = raw_rows[data_start_row:]

    if not data_rows or all(not any(pd.notna(cell) for cell in r) for r in data_rows):
        raise ValueError("❌ No usable data rows found")

    # === Step 3: Detect most common column shift ===
    def leading_blanks(row):
        return next((i for i, cell in enumerate(row) if str(cell).strip()), len(row))

    offsets = [leading_blanks(row) for row in data_rows if any(pd.notna(row))]
    common_offset = Counter(offsets).most_common(1)[0][0]

    if debug:
        print(f"🔍 Leading column offset detected: {common_offset}")

    # === Step 4: Normalize rows ===
    aligned_rows = []
    for row in data_rows:
        shift = common_offset if ENABLE_HEADER_SHIFTING else 0
        trimmed = row[shift:]
        if ENABLE_COLUMN_PADDING:
            trimmed = trimmed[:col_count] + [""] * max(0, col_count - len(trimmed))
        else:
            trimmed = trimmed[:col_count]
        aligned_rows.append([str(cell).strip() if cell is not None else "" for cell in trimmed])

    # === Step 5: Build DataFrame ===
    df = pd.DataFrame(aligned_rows, columns=gpt_headers)

    if df.shape[1] != col_count:
        raise ValueError(f"⚠️ Column count mismatch: expected {col_count}, got {df.shape[1]}")

    # === Step 6: Heuristic confidence check ===
    def token_density(row):
        return sum(bool(re.search(r"\w", str(x))) for x in row) / len(row)

    densities = [token_density(row) for row in aligned_rows]
    if np.mean(densities) < 0.4:
        raise ValueError(f"⚠️ Low data density (mean token density: {np.mean(densities):.2f})")

    if debug:
        print(f"✅ Final DataFrame shape: {df.shape}")

    return df


In [0]:
processed_gpt_flattened_data = {}

for sheet_key, raw_text in raw_text_map.items():
    print(f"\n🧠 Creating DataFrame using GPT-trusted headers for: {sheet_key}")
    try:
        file_name, sheet_name = sheet_key.split("::")

        # 🔄 Step 1: Copy from Unity Volume to local tmp
        volume_path = f"/Volumes/test/bronze/raw/{file_name}"
        file_path = copy_volume_file_to_tmp_via_spark(volume_path)

        # ✅ Step 2: Construct DataFrame using GPT + Smart parsing
        df = gpt_smart_dataframe_from_xlsx(
            file_path=file_path,
            sheet_name=sheet_name,
            raw_text=raw_text,
            client=client,
            debug=False  # Set to True for verbose output
        )

        processed_gpt_flattened_data[sheet_key] = df
        print(f"✅ Final DataFrame for {sheet_key}: {df.shape[0]} rows × {df.shape[1]} cols")

    except Exception as e:
        print(f"❌ Failed processing {sheet_key} — {e}")



🧠 Creating DataFrame using GPT-trusted headers for: Japanese_CHAR.xlsx::1.Statistics
✅ File copied to: /tmp/Japanese_CHAR.xlsx
✅ Final DataFrame for Japanese_CHAR.xlsx::1.Statistics: 58 rows × 8 cols

🧠 Creating DataFrame using GPT-trusted headers for: Japanese_CHAR.xlsx::Statistics Edited-USE
✅ File copied to: /tmp/Japanese_CHAR.xlsx
✅ Final DataFrame for Japanese_CHAR.xlsx::Statistics Edited-USE: 27 rows × 8 cols

🧠 Creating DataFrame using GPT-trusted headers for: Japanese_CHAR.xlsx::2 Buyer ex 200M
✅ File copied to: /tmp/Japanese_CHAR.xlsx
✅ Final DataFrame for Japanese_CHAR.xlsx::2 Buyer ex 200M: 53 rows × 12 cols

🧠 Creating DataFrame using GPT-trusted headers for: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1
✅ File copied to: /tmp/Japanese_CHAR.xlsx
✅ Final DataFrame for Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1: 54 rows × 12 cols

🧠 Creating DataFrame using GPT-trusted headers for: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 2
✅ File copied to: /tmp/Japanese_CHAR.xl

[Trace(request_id=tr-822db74bbf36447b82b09c66b12145e8), Trace(request_id=tr-7bfaf2bea6a84e2283ce79f10ff3948b), Trace(request_id=tr-5da8c5448242496aa5f59f5481dceccb), Trace(request_id=tr-8b7c7a53604943558cff30be723ce185), Trace(request_id=tr-d15a716689a04919b8ee1a9f5644fcd1), Trace(request_id=tr-84ecff4dbc1b40a981dfeff5c74d48a7), Trace(request_id=tr-7985ddec3c7b45d8a7ddf41c36bbaf86), Trace(request_id=tr-719f3af700ca44a48d331f3ece893d6f)]

In [0]:
from collections import defaultdict

def extract_values_from_dataframe(dataframe, columns, verbose=False, capture_extra_fields=True):
    """
    Extract column values from a structured pandas DataFrame, 
    including unexpected extra fields.

    Args:
        dataframe (pd.DataFrame): Cleaned and reconstructed sheet
        columns (List[dict]): GPT-inferred column metadata
        verbose (bool): Print missing/extra info
        capture_extra_fields (bool): If True, store unexpected columns in 'extra_fields'

    Returns:
        Dict[str, List[str]]: Extracted values by column including optional 'extra_fields'
    """
    extracted = {}
    expected_cols = {col.get("column_name") for col in columns if col.get("column_name")}
    df_cols = set(dataframe.columns)

    # === Step 1: Extract expected columns ===
    for col in expected_cols:
        if col not in df_cols:
            if verbose:
                print(f"⚠️ Column missing in DataFrame: {col}")
            extracted[col] = []
        else:
            extracted[col] = dataframe[col].astype(str).fillna("").tolist()

    # === Step 2: Extract extra fields ===
    if capture_extra_fields:
        extra_cols = df_cols - expected_cols
        if extra_cols and verbose:
            print(f"🔍 Extra fields detected: {extra_cols}")

        extracted["extra_fields"] = {
            col: dataframe[col].astype(str).fillna("").tolist()
            for col in extra_cols
        }

    return extracted


In [0]:
from collections import Counter
import pandas as pd

extracted_data_map = {}
failed_sheets = []

for sheet_key, raw_text in raw_text_map.items():
    print(f"\n📄 Extracting structured data from: {sheet_key}")

    try:
        # === Step 1: Extract GPT Schema ===
        schema = generate_rich_data_dictionary(raw_text, client)
        gpt_headers = schema.get("column_headers", [])
        data_start_row = schema.get("data_start_row", 0)
        semantic_columns = schema.get("columns", [])

        if not gpt_headers:
            raise ValueError("❌ GPT returned empty headers.")

        # ✅ Normalize column keys
        for col in semantic_columns:
            col["column_name"] = col.get("standardized_column_name") or col.get("original_column_name")

        # === Step 2: Semantic filtering (if enabled)
        if USE_SEMANTIC_FILTERING:
            filtered_columns = [
                col for col in semantic_columns
                if col.get("semantic_score", 0) >= semantic_score_threshold
            ]
            if len(filtered_columns) < 3:
                print(f"⚠️ Only {len(filtered_columns)} columns passed threshold. Using full set.")
                filtered_columns = semantic_columns
        else:
            filtered_columns = semantic_columns

        if not filtered_columns:
            raise ValueError("❌ No usable columns after filtering.")

        # === Step 3: Parse raw data rows ===
        lines = raw_text.strip().split("\n")
        rows = [line.split("\t") for line in lines[data_start_row:] if line.strip()]
        if not rows:
            raise ValueError("❌ No usable data rows found.")

        # === Step 4: Infer dominant structure ===
        row_lengths = [len(r) for r in rows if len(r) > 1]
        most_common_length = Counter(row_lengths).most_common(1)[0][0]

        # === Step 5: Smart left-shift detection ===
        leading_blanks = [
            next((i for i, cell in enumerate(row) if cell.strip()), len(row))
            for row in rows
        ]
        most_common_leading_empty = Counter(leading_blanks).most_common(1)[0][0]

        # === Step 6: Header realignment (if toggled)
        shift = most_common_leading_empty if ENABLE_HEADER_SHIFTING else 0
        if shift < len(gpt_headers):
            gpt_headers = gpt_headers[shift:]

        if len(gpt_headers) > most_common_length:
            gpt_headers = gpt_headers[:most_common_length]
        elif len(gpt_headers) < most_common_length:
            gpt_headers += [f"col_{i}" for i in range(len(gpt_headers), most_common_length)]

        # === Step 7: Normalize rows ===
        cleaned_rows = []
        extra_fields = []

        for row in rows:
            row_shift = shift if ENABLE_HEADER_SHIFTING and shift < len(row) else 0
            trimmed = row[row_shift:]

            # Capture overflow
            if ENABLE_EXTRA_FIELD_CAPTURE and len(trimmed) > most_common_length:
                extra_fields.append(trimmed[most_common_length:])
                trimmed = trimmed[:most_common_length]
            else:
                extra_fields.append([])

            # Pad if short
            if ENABLE_COLUMN_PADDING and len(trimmed) < most_common_length:
                trimmed += [""] * (most_common_length - len(trimmed))

            cleaned_rows.append(trimmed)

        # === Step 8: Create final DataFrame ===
        df = pd.DataFrame(cleaned_rows, columns=gpt_headers)
        if ENABLE_EXTRA_FIELD_CAPTURE:
            df["extra_fields"] = extra_fields

        print(f"🧠 GPT Headers: {gpt_headers}")
        print(f"✅ Filtered Columns: {[col['column_name'] for col in filtered_columns]}")
        print(f"✅ Final DF Columns: {df.columns.tolist()}")

        # === Step 9: Extract values ===
        values_by_column = extract_values_from_dataframe(df, filtered_columns)
        extracted_data_map[sheet_key] = values_by_column

        print(f"✅ Extracted {len(values_by_column)} columns for: {sheet_key}")

    except Exception as e:
        print(f"❌ Failed extracting {sheet_key} — {e}")
        failed_sheets.append(sheet_key)



📄 Extracting structured data from: Japanese_CHAR.xlsx::1.Statistics
🧠 GPT Headers: ['Underwriting_Year', 'Accident_Year', 'Gross_Premium_Written', 'Paid_Losses', 'Outstanding_Losses', 'Net_Balance', 'Incurred_Loss_Ratio', 'col_7', 'col_8']
✅ Filtered Columns: ['Cover_Type', 'Underwriting_Year', 'Accident_Year', 'Gross_Premium_Written', 'Paid_Losses', 'Outstanding_Losses', 'Net_Balance', 'Incurred_Loss_Ratio']
✅ Final DF Columns: ['Underwriting_Year', 'Accident_Year', 'Gross_Premium_Written', 'Paid_Losses', 'Outstanding_Losses', 'Net_Balance', 'Incurred_Loss_Ratio', 'col_7', 'col_8']
✅ Extracted 9 columns for: Japanese_CHAR.xlsx::1.Statistics

📄 Extracting structured data from: Japanese_CHAR.xlsx::Statistics Edited-USE
🧠 GPT Headers: ['Underwriting_Year', 'Accounting_Year', 'Gross_Premium_Written', 'Paid_Loss', 'Outstanding_Loss', 'Net_Balance', 'Incurred_Loss_Ratio', 'col_7']
✅ Filtered Columns: ['Cover_Type', 'Underwriting_Year', 'Accounting_Year', 'Gross_Premium_Written', 'Paid_Loss

[Trace(request_id=tr-87d8506d5bf64f4fba1d46b1f6e4be49), Trace(request_id=tr-5fa40140a10d45f0827237abe2df2ce6), Trace(request_id=tr-081603ac5eed45d8a2dc1daaef2415d2), Trace(request_id=tr-a2331110997b43ec9e8fcc60371d9415), Trace(request_id=tr-24dddffe540b44b3a590ec287c18150d), Trace(request_id=tr-9cced7b4f30d4027a2b30c3bbfbd1d49), Trace(request_id=tr-45770ee456f048bb9d708baac2d49baf), Trace(request_id=tr-e3f6b0774e474092bc4a4c742e98f425)]

In [0]:
from pyspark.sql import Row
import json

spark_dfs = {}

for sheet_key, col_data in extracted_data_map.items():
    try:
        print(f"\n🧱 Converting to Spark DataFrame: {sheet_key}")
        
        if not col_data or all(len(v) == 0 for v in col_data.values()):
            print(f"⚠️ Skipping empty: {sheet_key}")
            continue

        # Compute max length from valid lists
        max_len = max(len(v) for v in col_data.values() if isinstance(v, list))

        # Pad valid lists
        for col in col_data:
            if isinstance(col_data[col], list):
                col_data[col] += [""] * (max_len - len(col_data[col]))
            else:
                print(f"⚠️ Skipping non-list column: {col}")

        # Filter out anything that's not a valid column
        clean_col_data = {
            k: v for k, v in col_data.items()
            if isinstance(v, list) and len(v) == max_len
        }

        # Log dropped columns
        removed = set(col_data.keys()) - set(clean_col_data.keys())
        if removed:
            print(f"⚠️ Dropped non-list or misaligned columns: {removed}")

        df_pd = pd.DataFrame(clean_col_data)
        df_pd["file_name"] = sheet_key.split("::")[0]
        df_pd["sheet_name"] = sheet_key.split("::")[1]

        df_spark = spark.createDataFrame(df_pd)
        spark_dfs[sheet_key] = df_spark

        print(f"✅ Spark DataFrame: {sheet_key} → {df_spark.count()} rows")

    except Exception as e:
        print(f"❌ Failed converting {sheet_key} — {e}")



🧱 Converting to Spark DataFrame: Japanese_CHAR.xlsx::1.Statistics
⚠️ Skipping non-list column: extra_fields
⚠️ Dropped non-list or misaligned columns: {'extra_fields'}
✅ Spark DataFrame: Japanese_CHAR.xlsx::1.Statistics → 51 rows

🧱 Converting to Spark DataFrame: Japanese_CHAR.xlsx::Statistics Edited-USE
⚠️ Skipping non-list column: extra_fields
⚠️ Dropped non-list or misaligned columns: {'extra_fields'}
✅ Spark DataFrame: Japanese_CHAR.xlsx::Statistics Edited-USE → 23 rows

🧱 Converting to Spark DataFrame: Japanese_CHAR.xlsx::2 Buyer ex 200M
⚠️ Skipping non-list column: extra_fields
⚠️ Dropped non-list or misaligned columns: {'extra_fields'}
✅ Spark DataFrame: Japanese_CHAR.xlsx::2 Buyer ex 200M → 51 rows

🧱 Converting to Spark DataFrame: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1
⚠️ Skipping non-list column: extra_fields
⚠️ Dropped non-list or misaligned columns: {'extra_fields'}
✅ Spark DataFrame: Japanese_CHAR.xlsx::2 Buyers xs 200M - Edited 1 → 50 rows

🧱 Converting to Spark

In [0]:
def flag_data_quality_issues(df, sheet_key: str, threshold_null_pct: float = 0.8):
    """
    Flags columns with potential data quality issues.
    """
    issues = []
    row_count = df.count()
    if row_count == 0:
        return [{"sheet": sheet_key, "issue": "❌ Empty DataFrame"}]

    for field in df.schema.fields:
        name = field.name
        try:
            nulls = df.filter(col(name).isNull() | isnan(col(name))).count()
            null_pct = nulls / row_count

            if null_pct >= threshold_null_pct:
                issues.append({
                    "sheet": sheet_key,
                    "column": name,
                    "issue": f"⚠️ {int(null_pct * 100)}% nulls"
                })

            # Type check: numeric stored as string
            if field.dataType.simpleString() == "string":
                numeric_like = df.select(col(name)).rdd \
                    .map(lambda x: str(x[0]).replace(',', '').replace('.', '').isdigit()) \
                    .filter(lambda x: x).count()
                if numeric_like > 0 and numeric_like / row_count > 0.5:
                    issues.append({
                        "sheet": sheet_key,
                        "column": name,
                        "issue": "⚠️ Likely numeric, stored as string"
                    })

        except Exception as e:
            issues.append({
                "sheet": sheet_key,
                "column": name,
                "issue": f"❌ Error checking column — {str(e)}"
            })
    return issues


In [0]:
quality_issues = []
for sheet_key, df in spark_dfs.items():
    quality_issues.extend(flag_data_quality_issues(df, sheet_key))

if quality_issues:
    display(pd.DataFrame(quality_issues))
else:
    print("✅ No data quality issues found.")


sheet,column,issue
Japanese_CHAR.xlsx::1.Statistics,Accident_Year,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Incurred_Loss_Ratio,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Gross_Premium_Written,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Net_Balance,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Underwriting_Year,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Outstanding_Losses,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Paid_Losses,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,Cover_Type,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,file_name,❌ Error checking column — 'str' object is not callable
Japanese_CHAR.xlsx::1.Statistics,sheet_name,❌ Error checking column — 'str' object is not callable


In [0]:
# Pick the sheet you want to read (use the exact sheet key)
sheet_key = "Japanese_CHAR.xlsx::4 Risk Profile"  # Example

df = spark_dfs[sheet_key]
display(df)  # Show all rows


Credit_Limit_Band_300_499M_JPY,Credit_Limit_Band_500_999M_JPY,Credit_Limit_Band_50_99M_JPY,Credit_Limit_Band_246_299M_JPY,TDB_Score,Credit_Limit_Band_2000_2499M_JPY,Credit_Limit_Band_3000M_And_Above_JPY,Total_Buyers,Credit_Limit_Band_100_245M_JPY,Credit_Limit_Band_2500_2999M_JPY,Credit_Limit_Band_1500_1999M_JPY,Credit_Limit_Band_1000_1499M_JPY,file_name,sheet_name
1,1,10,6,75 or more,0,1,31,12,0,0,0,Japanese_CHAR.xlsx,4 Risk Profile
3,1,22,5,70-74,0,0,58,26,0,1,0,Japanese_CHAR.xlsx,4 Risk Profile
2,1,34,6,65-69,0,1,87,41,1,0,1,Japanese_CHAR.xlsx,4 Risk Profile
4,2,39,9,60-64,0,0,100,45,0,0,1,Japanese_CHAR.xlsx,4 Risk Profile
3,1,29,8,55-59,0,0,73,31,0,1,0,Japanese_CHAR.xlsx,4 Risk Profile
2,1,25,6,50-54,0,0,64,28,0,1,1,Japanese_CHAR.xlsx,4 Risk Profile
2,0,20,5,below 50,1,0,52,23,0,1,0,Japanese_CHAR.xlsx,4 Risk Profile
7,3,65,20,Not Scored,0,0,158,62,0,0,1,Japanese_CHAR.xlsx,4 Risk Profile
24,10,244,65,Total,1,2,623,268,1,4,4,Japanese_CHAR.xlsx,4 Risk Profile
,,,,,,(Limit in JPY Thousand),,,,,,Japanese_CHAR.xlsx,4 Risk Profile


In [0]:
from pyspark.sql import Row
import json

base_path = "/Volumes/bdx/data_dictionary/bdx_files"
spark_dfs = {}
quality_issues_log = []

for sheet_key, col_data in extracted_data_map.items():
    try:
        print(f"\n🧱 Converting to Spark DataFrame: {sheet_key}")

        if not col_data or all(len(v) == 0 for v in col_data.values()):
            print(f"⚠️ Skipping empty: {sheet_key}")
            continue

        max_len = max(len(v) for v in col_data.values())
        for col in col_data:
            col_data[col] += [""] * (max_len - len(col_data[col]))

        # ✅ Add extra_fields if toggle is enabled
        if ENABLE_EXTRA_FIELD_CAPTURE:
            # fallback if not already captured
            col_data["extra_fields"] = col_data.get("extra_fields", ["[]"] * max_len)

        df_pd = pd.DataFrame(col_data)

        file_name, sheet_name = sheet_key.split("::")
        df_pd["file_name"] = file_name
        df_pd["sheet_name"] = sheet_name

        df_spark = spark.createDataFrame(df_pd)
        spark_dfs[sheet_key] = df_spark

        # ✅ Quality Check
        if ENABLE_QUALITY_CHECK:
            issues = flag_data_quality_issues(df_spark, sheet_key)
            if issues:
                quality_issues_log.extend(issues)
                print(f"📉 Quality issues in {sheet_key}:")
                for i in issues:
                    print(f"  - {i['column'] if 'column' in i else 'unknown'}: {i['issue']}")

        # ✅ Write to Delta
        safe_sheet = sheet_name.replace(" ", "_").lower()
        output_path = f"{base_path}/{file_name}/{safe_sheet}"
        print(f"📁 Writing Delta table to: {output_path}")

        df_spark.write.format("delta").mode("overwrite").save(output_path)
        print(f"✅ Delta write complete for: {sheet_key}")

    except Exception as e:
        print(f"❌ Failed to convert/write {sheet_key} — {e}")


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-397567900857537>, line 4
      1 # Pick the sheet you want to read (use the exact sheet key)
      2 sheet_key = "test2.xlsx::Sheet1"  # Example
----> 4 df = spark_dfs[sheet_key]
      5 display(df)

KeyError: 'test2.xlsx::Sheet1'

In [0]:
if ENABLE_QUALITY_CHECK and quality_issues_log:
    try:
        df_issues = spark.createDataFrame(pd.DataFrame(quality_issues_log))
        df_issues.write.mode("overwrite").format("delta").saveAsTable("bdx.data_dictionary.quality_issues")
        print(f"✅ Data quality issues saved to Delta.")
    except Exception as e:
        print(f"❌ Failed to write quality issues to Delta: {e}")


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-397567900857537>, line 4
      1 # Pick the sheet you want to read (use the exact sheet key)
      2 sheet_key = "test2.xlsx::Sheet1"  # Example
----> 4 df = spark_dfs[sheet_key]
      5 display(df)

KeyError: 'test2.xlsx::Sheet1'

In [0]:
# for sheet_key, df in spark_dfs.items():
#     table_name = "extracted_" + sheet_key.replace(".xlsx", "").replace("::", "_").replace(" ", "_").lower()
#     df.write.format("delta").mode("overwrite").saveAsTable(table_name)
#     print(f"💾 Saved to Delta: {table_name}")

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-397567900857537>, line 4
      1 # Pick the sheet you want to read (use the exact sheet key)
      2 sheet_key = "test2.xlsx::Sheet1"  # Example
----> 4 df = spark_dfs[sheet_key]
      5 display(df)

KeyError: 'test2.xlsx::Sheet1'

In [ ]:

# ✅ Enhanced block extraction: support multi-row headers and column realignment
def extract_clean_block(df: pd.DataFrame, start: int, end: int, shift_headers: bool = True, pad_rows: bool = True) -> Tuple[pd.DataFrame, int]:
    block = df.iloc[start:end].reset_index(drop=True)

    # Combine first 2 non-null rows as header (if available)
    string_densities = block.apply(lambda row: row.dropna().apply(lambda x: isinstance(x, str)).sum(), axis=1)
    likely_header_rows = string_densities[string_densities > 3].index.tolist()
    header_rows = block.iloc[likely_header_rows[:2]] if len(likely_header_rows) >= 2 else block.iloc[[likely_header_rows[0]]]

    # Join headers with space or underscore
    headers = header_rows.fillna("").astype(str).agg(" ".join).str.strip().str.replace(" +", " ", regex=True)
    block.columns = headers
    clean_block = block.iloc[likely_header_rows[-1] + 1:].reset_index(drop=True)

    # Remove footer if 'total' in first column
    if 'Total' in clean_block.iloc[:, 0].astype(str).str.lower().values:
        clean_block = clean_block[~clean_block.iloc[:, 0].astype(str).str.lower().str.contains("total")]

    # Optional: pad rows to column count
    if pad_rows:
        clean_block = clean_block.apply(lambda row: row.tolist() + [None] * (len(headers) - len(row)) if len(row) < len(headers) else row, axis=1, result_type="expand")
        clean_block.columns = headers

    return clean_block, likely_header_rows[-1]
